# What a Histogram Quantile Actually Computes

Companion notebook for the [PromQL and Recording Rules](https://codemia.io/courses/observability_and_delivery/obs_promql_and_recording_rules) lesson on Codemia.

**What you will build**: Prometheus histogram_quantile from first principles. You will implement the interpolation
Prometheus uses, measure how wrong it is on three bucket layouts, watch a reported p99 pin itself to
a bucket boundary while the real one keeps climbing, see why averaging percentiles across instances
is arithmetic nobody should trust, and price the recording rule that makes the correct query cheap.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(31)

N = 400_000          # requests observed in one scrape interval's worth of traffic

## Concept recap

A classic Prometheus histogram is not a distribution. It is a handful of counters, one per bucket
boundary, each counting requests whose latency was less than or equal to that boundary. The raw data
is gone; what survives is a cumulative step function with a dozen steps in it.

`histogram_quantile(0.99, ...)` takes those counters, finds the bucket the ninety ninth percentile
falls into, and interpolates linearly between that bucket's boundaries. Every property people find
surprising about it follows from those two facts: the answer can only be as precise as the bucket
layout allows, and inside a bucket it assumes requests are spread evenly, which they never are.

None of that makes histograms the wrong tool. They are the only latency signal that aggregates
correctly across a fleet, which is more than can be said for a precomputed percentile. But you have
to know what the number means.

## A latency population with a shape

In [ ]:
# Most requests hit cache. A few percent miss and do real work. This bimodality is
# the normal case, not a pathology, and it is what bucket layouts handle badly.
slow = rng.random(N) < 0.03
latency = np.where(slow,
                   rng.lognormal(np.log(1.2), 0.5, N),      # seconds
                   rng.lognormal(np.log(0.09), 0.6, N))

QUANTILES = [0.5, 0.9, 0.95, 0.99, 0.999]
true_q = {q: float(np.quantile(latency, q)) for q in QUANTILES}

print(f'{"quantile":>10}{"true latency":>16}')
for q in QUANTILES:
    print(f'{q:>10.3f}{true_q[q] * 1000:>13.0f} ms')
print()
print(f'mean {latency.mean() * 1000:.0f} ms, which is between the two modes and describes neither')
print(f'{slow.mean():.0%} of requests do real work; they are '
      f'{np.median(latency[slow]) / np.median(latency[~slow]):.0f}x slower than the rest')

## The three layouts, and the interpolation Prometheus performs

In [ ]:
GO_DEFAULT = [.005, .01, .025, .05, .1, .25, .5, 1, 2.5, 5, 10]     # client_golang's default
TUNED = [0.002 * 2 ** k for k in range(13)]                          # 2 ms doubling to 8.2 s
COARSE = [0.1, 0.5, 1, 5]                                            # chosen by hand, in a hurry
FINE = [0.001 * 1.1 ** k for k in range(int(np.log(60_000) / np.log(1.1)) + 1)]

LAYOUTS = {'client_golang default': GO_DEFAULT, 'tuned, 2x from 2 ms': TUNED,
           'coarse, 4 buckets': COARSE, 'fine, 1.1x growth': FINE}


def observe(values, bounds):
    """The counters a Prometheus client would export: cumulative counts, plus +Inf."""
    cum = np.searchsorted(np.sort(values), bounds, side='right').astype(float)
    return np.append(cum, float(len(values)))


def histogram_quantile(q, bounds, cum):
    """Prometheus' own algorithm, including its two edge cases."""
    total = cum[-1]
    if total == 0:
        return float('nan')
    rank = q * total
    i = int(np.searchsorted(cum, rank, side='left'))
    if i == len(cum) - 1:                 # falls in +Inf: clamped to the last finite bound
        return bounds[-1]
    lower = 0.0 if i == 0 else bounds[i - 1]
    upper = bounds[i]
    count_below = 0.0 if i == 0 else cum[i - 1]
    width = cum[i] - count_below
    if width <= 0:
        return upper
    return lower + (upper - lower) * (rank - count_below) / width


print(f'{"layout":>24}{"buckets":>9}' + ''.join(f'{"p" + format(q * 100, "g"):>12}' for q in QUANTILES))
print(f'{"":>24}{"":>9}{"reported / true":>60}')
for name, bounds in LAYOUTS.items():
    cum = observe(latency, bounds)
    cells = ''
    for q in QUANTILES:
        est = histogram_quantile(q, bounds, cum)
        cells += f'{est / true_q[q]:>11.2f}x'
    print(f'{name:>24}{len(bounds) + 1:>9}{cells}')
print()
print(f'{"layout":>24}' + ''.join(f'{"p" + format(q * 100, "g"):>12}' for q in QUANTILES)
      + '   reported, ms')
for name, bounds in LAYOUTS.items():
    cum = observe(latency, bounds)
    cells = ''.join(f'{histogram_quantile(q, bounds, cum) * 1000:>9.0f} ms' for q in QUANTILES)
    print(f'{name:>24}{cells}')
print(f'{"true":>24}' + ''.join(f'{true_q[q] * 1000:>9.0f} ms' for q in QUANTILES))

Read the first table as a correction factor. A ratio of 1.00 means the histogram told the truth.

The default Go layout reports a p99 about a quarter too high, because the ninety ninth percentile
lands in the bucket between one and two and a half seconds and gets interpolated across a width of
one and a half seconds. The p99.9 is worse, and in the coarse layout both are pure fiction: with a
boundary at five seconds and nothing between one and five, the interpolation has no information to
work with and invents a number in the middle.

The fine layout, each bucket ten percent wider than the last, is within a couple of percent
everywhere. That is the arithmetic behind native histograms: the error of an interpolated quantile is
bounded by the relative width of the bucket it lands in, so a constant growth factor gives a constant
relative error at every latency, which is exactly what you want from a latency metric and is not what
a hand written list of boundaries gives you.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.8))

xs = np.logspace(np.log10(0.002), np.log10(12), 400)
ax1.plot(xs * 1000, [np.mean(latency <= x) for x in xs], color='black',
         linewidth=2, label='true distribution')
for name, bounds, colour in [('client_golang default', GO_DEFAULT, 'tab:blue'),
                             ('coarse, 4 buckets', COARSE, 'tab:red')]:
    cum = observe(latency, bounds)
    ax1.step(np.array(bounds) * 1000, cum[:-1] / cum[-1], where='post',
             color=colour, label=name)
    ax1.plot(np.array(bounds) * 1000, cum[:-1] / cum[-1], 'o', color=colour, markersize=4)
ax1.axhline(0.99, color='grey', linestyle=':', label='p99')
ax1.set_xscale('log')
ax1.set_xlabel('latency (ms)')
ax1.set_ylabel('share of requests at or below')
ax1.set_title('A histogram is the dots. Everything between them is assumed.')
ax1.legend(loc='upper left', fontsize=8)
ax1.grid(alpha=0.3)

for name, bounds in LAYOUTS.items():
    cum = observe(latency, bounds)
    qs = np.linspace(0.5, 0.999, 120)
    err = [histogram_quantile(q, bounds, cum) / np.quantile(latency, q) for q in qs]
    ax2.plot(qs, err, label=name)
ax2.axhline(1.0, color='black', linestyle='--')
ax2.set_ylim(0, 3)
ax2.set_xlabel('quantile')
ax2.set_ylabel('reported / true')
ax2.set_title('Error is a property of the layout, not of the quantile')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)
plt.show()

The left panel is the honest picture of what a classic histogram stores: eleven or four points on a
curve. `histogram_quantile` draws the straight line between two of those points and reports where it
crosses. The right panel is the consequence, and the sawtooth is the tell. Error is not a smooth
function of the quantile you asked for; it spikes wherever a quantile lands near the wide end of a
bucket and falls back to nothing wherever a bucket boundary happens to sit close to the truth.

That is why comparing a p99 across two services with different bucket layouts is meaningless, and why
changing a layout changes your reported latency without anything in production changing at all.

## The failure that looks like a plateau

In [ ]:
print('a dependency gets slower. the true p99 climbs. what does each layout report?')
print()
print(f'{"regression":>12}{"true p99":>12}'
      + ''.join(f'{n.split(",")[0][:14]:>18}' for n in LAYOUTS))
for factor in [1, 1.5, 2, 3, 5, 10, 20]:
    shifted = latency * factor
    truth = np.quantile(shifted, 0.99)
    cells = ''
    for name, bounds in LAYOUTS.items():
        cum = observe(shifted, bounds)
        est = histogram_quantile(0.99, bounds, cum)
        pinned = ' (pinned)' if abs(est - bounds[-1]) < 1e-12 else ''
        cells += f'{f"{est:.2f} s{pinned}":>18}'
    print(f'{factor:>11.1f}x{truth:>10.2f} s{cells}')
print()
over = {name: float(np.mean(latency * 20 > bounds[-1])) for name, bounds in LAYOUTS.items()}
for name, frac in over.items():
    print(f'at 20x, {frac:>6.2%} of requests land in the +Inf bucket of {name}')

This is the behaviour worth recognising on a dashboard at three in the morning. Once the quantile
falls into the `+Inf` bucket, `histogram_quantile` has nothing above the last finite boundary to
interpolate against, so Prometheus returns that boundary. The graph flattens into a horizontal line
at a suspiciously round number while the service keeps getting worse.

A p99 sitting exactly on a bucket boundary is not a stable service. It is a histogram that has run
out of range, and it is the reason the top boundary of your layout should be above anything you would
ever tolerate rather than above anything you have ever seen. The coarse layout plateaus almost
immediately; the fine layout keeps reporting the truth throughout, because its top boundary is a
minute.

## Why avg(p99) is not a p99

In [ ]:
INSTANCES = 20
BOUNDS = FINE          # the layout error is already measured; use a layout without one

def make_fleet(weights, slowdown):
    """Simulate one scrape interval across a fleet and aggregate it four ways."""
    counts = (weights / weights.sum() * N).astype(int)
    summed = np.zeros(len(BOUNDS) + 1)
    every_request, reported = [], []
    for i in range(INSTANCES):
        n = counts[i]
        s = rng.random(n) < 0.03
        lat = np.where(s, rng.lognormal(np.log(1.2), 0.5, n),
                       rng.lognormal(np.log(0.09), 0.6, n)) * slowdown[i]
        cum = observe(lat, BOUNDS)
        summed += cum
        every_request.append(lat)
        reported.append(histogram_quantile(0.99, BOUNDS, cum))
    fleet = np.concatenate(every_request)
    reported = np.array(reported)
    return {'true': float(np.quantile(fleet, 0.99)),
            'summed': histogram_quantile(0.99, BOUNDS, summed),
            'avg': float(reported.mean()), 'max': float(reported.max()),
            'share': float(counts[np.argmax(slowdown)] / counts.sum())}


even = np.ones(INSTANCES)
uneven = rng.uniform(0.4, 1.6, INSTANCES)

healthy = np.ones(INSTANCES)

busy_slow = np.ones(INSTANCES)
busy_slow[7] = 4.0
busy_weights = uneven.copy()
busy_weights[7] = uneven.sum() / 3            # that instance takes a quarter of the traffic

quiet_slow = np.ones(INSTANCES)
quiet_slow[[2, 11, 16]] = 6.0
quiet_weights = uneven.copy()
quiet_weights[[2, 11, 16]] = uneven.sum() * 0.01

SCENARIOS = [
    ('all instances healthy', uneven, healthy),
    ('one busy instance 4x slow', busy_weights, busy_slow),
    ('three quiet instances 6x slow', quiet_weights, quiet_slow),
]

print(f'{"fleet":>32}{"true p99":>11}{"sum(buckets)":>15}'
      f'{"avg(p99)":>11}{"max(p99)":>11}{"avg / true":>12}')
for name, w, sd in SCENARIOS:
    r = make_fleet(w, sd)
    print(f'{name:>32}{r["true"]:>9.2f} s{r["summed"]:>13.2f} s'
          f'{r["avg"]:>9.2f} s{r["max"]:>9.2f} s{r["avg"] / r["true"]:>11.2f}x')
print()
r = make_fleet(busy_weights, busy_slow)
print(f'in the second fleet the slow instance serves {r["share"]:.0%} of the traffic, '
      f'so the fleet p99 moves and the average barely does.')
r = make_fleet(quiet_weights, quiet_slow)
print(f'in the third, the three slow instances serve {r["share"]:.1%} of the traffic each, '
      f'so the average moves and the fleet p99 barely does.')

Summing the buckets first and taking the quantile of the sum tracks the true fleet p99 in all three
fleets, which is what a correct aggregation looks like. The average of the per instance p99s agrees
with it in the first fleet and then goes wrong in both directions, which is worse than being
consistently wrong, because the first row is what people check before trusting the metric.

The mechanism is in the last two lines of output. An average weights every instance equally, while a
percentile of the fleet weights every request equally. When the slow instance is a busy one the
average divides its badness by twenty and under-reports the fleet by a quarter; when the slow
instances are nearly idle the average inflates the fleet by sixty percent on the strength of three
percent of the traffic. Neither number is a percentile of anything, because a percentile is not a
linear function of the data and averaging one is not an operation.

The two aggregations worth having on a dashboard are therefore `histogram_quantile` over summed
buckets, which is the fleet's latency, and `max` or `topk` over per instance quantiles, which is how
you find the one bad instance. The `max` column spots the outlier in both broken fleets; the average
spots it in neither.

In PromQL that distinction is a sum inside the function rather than outside it:

    histogram_quantile(0.99, sum by (le) (rate(http_request_duration_seconds_bucket[5m])))

Everything hinges on `by (le)`: drop it and you aggregate away the bucket boundaries, keep any other
label in it and you get a quantile per value of that label rather than one for the fleet.

## What the correct query costs, and the rule that fixes it

In [ ]:
ENDPOINTS, INSTANCE_COUNT = 40, 120
SCRAPE_S, WINDOW_S = 15, 300
SAMPLES_PER_SERIES = WINDOW_S // SCRAPE_S
PANELS, STEPS = 6, 60

bucket_series = ENDPOINTS * INSTANCE_COUNT * (len(GO_DEFAULT) + 1)
rule_series = ENDPOINTS * (len(GO_DEFAULT) + 1)
raw_eval = bucket_series * SAMPLES_PER_SERIES
rule_eval = rule_series * SAMPLES_PER_SERIES

print(f'bucket series for one histogram metric   {bucket_series:>14,}')
print(f'samples read by one raw evaluation       {raw_eval:>14,}')
print(f'series written by the recording rule     {rule_series:>14,}')
print(f'samples read by one evaluation of those  {rule_eval:>14,}')
print(f'ratio                                    {raw_eval / rule_eval:>13,.0f}x')
print()

print(f'one dashboard load, {PANELS} panels x {STEPS} points:')
print(f'  straight from buckets {raw_eval * PANELS * STEPS:>16,} samples')
print(f'  from the rule         {rule_eval * PANELS * STEPS:>16,} samples')
print()

print('what the rule costs to maintain, and when it pays for itself:')
print(f'{"rule interval":>16}{"samples read/day":>20}{"break-even loads/day":>24}')
for interval in [15, 30, 60, 120]:
    per_day = raw_eval * 86_400 // interval
    saved_per_load = (raw_eval - rule_eval) * PANELS * STEPS
    print(f'{interval:>14} s{per_day:>20,}{per_day / saved_per_load:>24.1f}')
print()

print('but a dashboard is rarely the only consumer. one alert on the same expression,')
print(f'evaluated every {SCRAPE_S} s, reads {raw_eval * 86_400 // SCRAPE_S:,} samples a day on its own,')
print(f'or {rule_eval * 86_400 // SCRAPE_S:,} if it reads the rule. the rule costs the same as the alert')
print('it replaces, so the first alert makes it free and every other consumer is profit.')
print()
print('storage for the pre-aggregated series, at 1.7 bytes per sample:')
print(f'  {rule_series * 86_400 / SCRAPE_S * 1.7 / 2 ** 20:>8.1f} MB/day'
      f'   {rule_series * 86_400 * 15 / SCRAPE_S * 1.7 / 2 ** 30:>6.2f} GB over 15 days')

A recording rule is not a cache, it is a change of shape: it evaluates the expensive aggregation on a
schedule and writes a much smaller series that dashboards and alerts read instead. The reduction is
exactly the instance count, because that is the dimension the rule aggregated away.

The break even table is the part usually left out of the advice. A rule evaluating every fifteen
seconds reads billions of samples a day whether anyone looks or not, so measured against dashboard
traffic alone it needs a double figure number of loads a day to be worth it, and a longer evaluation
interval lowers that bar in proportion.

The argument that actually settles it is the paragraph below the table. Alerting rules evaluate
continuously too, so an alert reading raw buckets already pays the rule's cost without producing a
reusable series. Once a single alert exists on the expression, the recording rule is free and every
dashboard, every other alert and every ad hoc query against it is pure saving. A rule for an
expression with no alert and no readers is the one to delete.

Two practical notes that follow from the shape of the data rather than from taste. The rule must
aggregate `by (le)` and keep every label a consumer might need, because what the rule drops is gone
for good at the resolution the rule produces. And alerts should read rule output rather than raw
buckets, so that the alert and the dashboard a responder opens are computing the same number from the
same series.

## What to take away

- A classic histogram stores a dozen points on a cumulative curve. `histogram_quantile` interpolates
  linearly between two of them, so the reported quantile is only as good as the bucket it lands in.
- Error is bounded by the relative width of that bucket. Constant growth factors give constant
  relative error, which is the whole argument for native histograms over hand written boundaries.
- A quantile that falls in `+Inf` is reported as the last finite boundary. A p99 that flattens onto a
  round number is a layout that has run out of range, not a service that has stabilised.
- Pick the top boundary from what you refuse to tolerate, not from what you have measured so far.
- Sum buckets first, then take the quantile. `avg` of per instance percentiles weights instances by
  nothing in particular and hides single instance failures; use `max` or `topk` to find those.
- Recording rules turn the correct fleet query from a scan of every instance's buckets into a read of
  a few hundred series, and they pay for themselves at a couple of dashboard loads a day.